# RegDet V1.1 — HMM SHARE label charts

**One job:** show the same 2-hour Nifty price history labelled five times, at five
different *HMM shares*, under identical visual treatment, so the labelling can be
judged **by eye**.

    HMM_share = 100 * (1 - BAR_DIR_WEIGHT)

| HMM share | `BAR_DIR_WEIGHT` | what it means |
|---|---|---|
| **100%** | `w = 0.00` | direction is 100% HMM state mass — **the current shipped default** |
| **75%**  | `w = 0.25` | |
| **50%**  | `w = 0.50` | |
| **25%**  | `w = 0.75` | the arm the user says looked best |
| **0%**   | `w = 1.00` | no HMM in direction at all — pure causal per-bar momentum |

**HMM SHARE is the primary identifier everywhere** in this notebook — every title,
legend entry, filename and table row. `w=` is shown second, in brackets. Do not
read the two the other way round: they run in opposite directions.

### What makes this a fair comparison
1. **ONE shared HMM ensemble fit**, fitted once, reused by all five arms. The five
   arms differ *only* by `BAR_DIR_WEIGHT`. The model objects are asserted
   bit-identical before and after all five labelling passes.
2. **Identical visual treatment**: one regime→colour map, one date range per figure,
   one y-axis limit pair per figure (asserted equal across its five panels), one
   figure width, one linewidth. A different y-scale alone changes the visual verdict.
3. **Strictly causal labels.** A label at bar `t` reads only bars `<= t`. This is an
   inherited property of the engine, inlined verbatim and not modified here.

### What this notebook deliberately does NOT do
No six-family scorecard. No walk-forward. No A/B ladder. No w-sweep. **No forward
returns, no Sharpe, no economic metric of any kind** — this is about *regime
description accuracy*, not profitability, and a profit number is not smuggled in as
a tiebreaker. **No winner is declared.** The five pictures are evidence; the choice
is the user's.

### What to do with it
Run top to bottom (well under a minute). Five PNGs are written to the working
directory and their filenames printed at the end — export those and send them back.
The final cell prints a complete, copy-pasteable config block that reproduces
exactly what the charts show.

In [ ]:
%pip install -q hmmlearn yfinance

## 1. Engine, inlined verbatim

Inlined out of `build_master_notebook_v2.py` rather than imported, because a
standalone `.ipynb` on Kaggle cannot import a local `.py` next to it. Source:
`build_master_notebook_v2.py cell 3 (constants/imports), cell 5 (load_2h/_synth), cell 7 (feature + labeling engine), cell 9 (plot helpers), cell 76 (W_SECONDARY_* fidelity metric)`. Not one line is modified — same constants, same features, same
gates, same hysteresis, same causality. The master notebook and its generator are
untouched by this notebook.

In [ ]:
# ==========================================================================
# CONSTANTS + IMPORTS -- INLINED VERBATIM from build_master_notebook_v2.py
# (master notebook code cell 3). Unmodified.
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier (1.0 = production windows)

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = 9      # bars over which efficiency is measured (~ TREND_FEATURE horizon)
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.0 -- FIX 4 IS RETAINED AS A SWITCH BUT SET OFF.
#
# It was 0.75 (and before that 0.5). It is now 0.0. The machinery, the sweep in
# 7H-vi and the overlay in 7H-vii all stay; only the shipped weight moved.
#
# WHY IT WAS TURNED OFF. Scored across three real-data runs, the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.0
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=6 was the size the
# offline study measured (direction-call agreement between independent pools
# 81.9% single -> 91.9% at K=6), and K=6 is now the ADOPTED value: it is the size
# the stability study actually measured, and the cost is linear. The earlier K=4
# compromise existed only because Section 5d re-ran the ENTIRE walk-forward in
# both arms; 5d is OFF by default now (RUN_SEED_STABILITY=False below), so the
# runtime argument for K=4 no longer applies.
ENSEMBLE_K     = 6
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5, VOL_SLOW=20, SWING_WIN=20)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'vol_norm'    # 'frozen_z' = pre-change | 'vol_norm' = adopted
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'block'   # 'block' (adopted) | 'allow' (pre-change) | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON. NOTE: w={BAR_DIR_WEIGHT} is NOT the shipped "
          "default (0.0); it broke")
    print("              the direction-level forward-return ordering on real data.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# DATA LOADER -- INLINED VERBATIM (master code cell 5).
# yfinance 60m -> 2h resample, with a synthetic fallback. Unmodified.
# ==========================================================================
def load_2h():
    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""
    try:
        import yfinance as yf

        def h2(tk):
            raw = yf.download(tk, interval='60m', period='730d',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index
            return c.resample('2h').last().dropna()

        nifty = h2('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = h2('^INDIAVIX')
            vix = (vix.reindex(nifty.index).ffill().bfill()
                   if vix is not None and len(vix) else pd.Series(15.0, index=nifty.index))
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'yfinance 60m->2h: {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few bars')
    except Exception as e:
        print(f'yfinance unavailable ({e}); falling back to synthetic 2h data.')
        return _synth()


def _synth():
    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),
           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),
           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),
           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]
    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []
    for i in range(n):
        f = i / n
        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, q in REG:
            if fs <= f < fe:
                mu, sg, vm, vf = m, s, v, q
                break
        lvl *= np.exp(rng.normal(mu, sg))
        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nv.append(lvl); vv.append(pv)
    return (pd.Series(nv, index=idx, name='nifty'),
            pd.Series(vv, index=idx, name='vix'), True)


nifty, vix, TAC_SYNTH = load_2h()
if TAC_SYNTH:
    print(f"\n*** TAC_SYNTH = True  ->  SYNTHETIC data ({len(nifty)} bars). "
          f"Results below are ILLUSTRATIVE ONLY. ***\n")
else:
    print(f"\n*** TAC_SYNTH = False  ->  REAL yfinance data ({len(nifty)} bars). "
          f"Results below are decision-grade. ***\n")
print(f"Span: {nifty.index[0]}  ->  {nifty.index[-1]}")

In [ ]:
# ---------------------------------------------------------------------------
# PROVENANCE BANNER. Printed loudly and early so a sandbox run can never be
# mistaken for a decision-grade one.
# ---------------------------------------------------------------------------
if TAC_SYNTH:
    print('#' * 100)
    for _ in range(3):
        print('#   SYNTHETIC - ILLUSTRATIVE ONLY   ' * 2)
    print('#' * 100)
    print('#  The yfinance fetch FAILED (it is firewalled in some sandboxes).')
    print('#  Every chart and number below is generated from a synthetic GBM series.')
    print('#  It is a PLUMBING TEST ONLY. It says NOTHING about real Nifty regimes.')
    print('#  Re-run on Kaggle, where yfinance works, before judging any labelling.')
    print('#' * 100)
else:
    print('=' * 100)
    print('REAL yfinance data - decision-grade. Charts below are of actual Nifty 2h bars.')
    print('=' * 100)
print(f'bars={len(nifty)}   span {nifty.index[0]} -> {nifty.index[-1]}')

In [ ]:
# ==========================================================================
# FEATURE + LABELING ENGINE -- INLINED VERBATIM (master code cell 7).
# build_features / intensity gates / direction buckets / bar_direction_masses /
# fit_hmm_ensemble / label_bars. Causality and every default unmodified.
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    n_side = max(1, round(n_st / 5))
    n_bear = (n_st - n_side) // 2
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# PLOT HELPERS -- INLINED VERBATIM (master code cell 9).
# regime_blocks / shade_bands (the batched, autolim-safe regime bands) /
# regime_legend. Unmodified.
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i - 1]))
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

In [ ]:
# ==========================================================================
# DESCRIPTIVE FIDELITY -- INLINED VERBATIM (master code cell 76),
# the w-sweep SECONDARY metric. NOT reinvented here. Strictly causal:
# agreement of the emitted direction at t with sign(C[t] - C[t-k]), k=9.
# Only the two functions are taken -- none of the sweep machinery.
# ==========================================================================
W_SECONDARY_K = 9             # headline trailing lookback, ~3 days of 2h bars
W_SECONDARY_METRIC_NAME = ('fraction of non-SIDEWAYS bars where sign(direction label) '
                           '== sign(C[t] - C[t-k]), k=%d, STRICTLY CAUSAL' % W_SECONDARY_K)


def W_SECONDARY_PERBAR(labels, close_arr, k=None):
    # Per-bar agreement between the emitted direction and the TRAILING move.
    # Returns (agree_bool_array, scored_mask), both length n.
    #
    # STRICTLY CAUSAL BY CONSTRUCTION: entry t reads labels[t], close[t] and
    # close[t-k] -- indices <= t only. Nothing in this function indexes forward.
    k = W_SECONDARY_K if k is None else int(k)
    labels = np.asarray(labels, dtype=object)
    close_arr = np.asarray(close_arr, dtype=float)
    n = len(close_arr)
    lab_sign = np.where(np.isin(labels, ['H_BULL', 'L_BULL']), 1.0,
                        np.where(np.isin(labels, ['H_BEAR', 'L_BEAR']), -1.0, 0.0))
    trail = np.full(n, np.nan)
    if k < n:
        trail[k:] = close_arr[k:] - close_arr[:n - k]   # C[t] - C[t-k]
    move_sign = np.sign(trail)
    m = (lab_sign != 0) & np.isfinite(move_sign) & (move_sign != 0)
    return (lab_sign == move_sign), m


def W_SECONDARY_METRIC(labels, close_arr, k=None):
    # DESCRIPTIVE FIDELITY -- how well the labels describe the move that has
    # ALREADY HAPPENED. This is NOT predictive skill and must never be quoted as
    # validation; see the prose above for why it is partly mechanical at high w.
    #
    # SIDEWAYS bars are excluded (no directional claim); bars t < k are excluded
    # (no lookback available yet).
    #
    # Returns (agreement_fraction, n_scored).
    agree, m = W_SECONDARY_PERBAR(labels, close_arr, k)
    if m.sum() == 0:
        return np.nan, 0
    return float(np.mean(agree[m])), int(m.sum())

## 2. The five arms — the mapping, asserted in code

Getting `HMM share` and `w` backwards would invert the whole conclusion, so the
identity `HMM_share = 100 * (1 - BAR_DIR_WEIGHT)` is asserted rather than trusted.

In [ ]:
# ---------------------------------------------------------------------------
# THE FIVE ARMS. HMM SHARE is the primary identifier; w is secondary.
#   HMM_share = 100 * (1 - BAR_DIR_WEIGHT)
# Ordered 100% HMM first (most HMM) down to 0% HMM (no HMM in direction).
# ---------------------------------------------------------------------------
ARM_SHARES = [100, 75, 50, 25, 0]                 # percent HMM
ARM_W = [1.0 - s / 100.0 for s in ARM_SHARES]     # BAR_DIR_WEIGHT per arm
ARMS = list(zip(ARM_SHARES, ARM_W))

# ASSERT THE MAPPING BOTH WAYS. This is the one error that would invert the verdict.
for _s, _w in ARMS:
    assert abs(_s - 100.0 * (1.0 - _w)) < 1e-12, f'share/w mapping broken: {_s} vs w={_w}'
    assert abs(_w - (1.0 - _s / 100.0)) < 1e-12
assert ARM_W[0] == 0.0 and ARM_SHARES[0] == 100, '100% HMM must be w=0.00'
assert ARM_W[-1] == 1.0 and ARM_SHARES[-1] == 0, '0% HMM must be w=1.00'
assert sorted(ARM_SHARES, reverse=True) == ARM_SHARES, 'arms must descend in HMM share'
assert BAR_DIR_WEIGHT == 0.0, 'shipped default expected to be w=0.00 (=100% HMM)'
SHIPPED_SHARE = int(round(100.0 * (1.0 - BAR_DIR_WEIGHT)))


def arm_id(share, w):
    # THE canonical arm label. HMM share first, w second, everywhere.
    tag = '  [SHIPPED DEFAULT]' if w == BAR_DIR_WEIGHT else ''
    note = ''
    if w == 1.0:
        note = '  no HMM in direction'
    elif w == 0.0:
        note = '  direction 100% HMM'
    return f'{share:3d}% HMM  (w={w:.2f}){note}{tag}'


def arm_slug(share, w):
    return f'hmm{share:03d}_w{w:.2f}'.replace('.', 'p')


print('=' * 100)
print('THE FIVE ARMS      HMM_share = 100 * (1 - BAR_DIR_WEIGHT)      mapping ASSERTED OK')
print('=' * 100)
for _s, _w in ARMS:
    print('   ' + arm_id(_s, _w))
print()
print(f'shipped default in this notebook = {SHIPPED_SHARE}% HMM (w={BAR_DIR_WEIGHT:.2f})')
print('Only BAR_DIR_WEIGHT differs between arms. Everything else is byte-identical.')
print()
print('SCOPE LIMIT, stated up front: w touches the DIRECTION axis ONLY. The H-vs-L')
print('INTENSITY axis contains zero HMM in this engine, so no arm here can change')
print('how H/L is decided -- only bull/side/bear.')

## 3. ONE shared HMM fit

One `StandardScaler`, one anchored ensemble fit on the leading `TRAIN_FRACTION` of
bars, seeds `BASE_SEED + 0..ENSEMBLE_K-1`. Every arm below reuses *these* model
objects. Nothing is refitted per arm, so any visual difference between panels is
attributable to `BAR_DIR_WEIGHT` and to nothing else.

In [ ]:
import time as _time
T0 = _time.time()

# ---------------------------------------------------------------------------
# THE SINGLE SHARED FIT. Adopted config, pinned exactly as the master pins it.
# ---------------------------------------------------------------------------
CFG = next(c for c in CONFIGS if c['name'] == ADOPTED_CONFIG_NAME)
FEAT = list(CFG['features'])

fdf = build_features(nifty, vix, LOOKBACK_SCALE)
X_raw = fdf[FEAT].values
DATES = fdf.index
N_FIT = max(int(len(X_raw) * TRAIN_FRACTION), 50)
SCALER = StandardScaler().fit(X_raw[:N_FIT])
XS = SCALER.transform(X_raw)
TREND_RAW = fdf[TREND_FEATURE].values
CLOSE = nifty.reindex(DATES).values.astype(float)

_t = _time.time()
MODELS, ALL_CONV = fit_hmm_ensemble(XS[:N_FIT], CFG['N'], CFG['cov'])
FIT_SECS = _time.time() - _t
FIT_SEEDS = ensemble_seeds()

# Fingerprint the fit so the "shared, never refitted" claim can be ASSERTED, not
# asserted-by-comment. Raw bytes, not a tolerance.
def fit_fingerprint(models):
    return [(m.means_.tobytes(), m.transmat_.tobytes(), m.startprob_.tobytes())
            for m in models]

FIT_FP_BEFORE = fit_fingerprint(MODELS)

print('=' * 100)
print('ONE SHARED HMM FIT - reused by all five arms' + synth_tag())
print('=' * 100)
print(f'  config          {ADOPTED_CONFIG_NAME!r}   N_STATES={CFG["N"]}  cov={CFG["cov"]!r}')
print(f'  features ({len(FEAT)})   {FEAT}')
print(f'  ensemble        ENSEMBLE_K={len(MODELS)}  BASE_SEED={BASE_SEED}  seeds={FIT_SEEDS}')
print(f'  all converged   {ALL_CONV}')
print(f'  anchored fit    {N_FIT}/{len(XS)} bars ({TRAIN_FRACTION:.0%})   '
      f'scaler fit on the same leading window')
print(f'  bars labelled   {len(DATES)}   {DATES[0]} -> {DATES[-1]}')
print(f'  fit time        {FIT_SECS:.2f}s')
print()
print('  NO ARM REFITS. Five label passes over these exact model objects.')

## 4. Five label passes over the shared fit

In [ ]:
# ---------------------------------------------------------------------------
# FIVE LABEL PASSES. Identical call, identical fit, identical everything --
# bar_dir_weight is the only argument that moves.
# ---------------------------------------------------------------------------
LABELS = {}          # share -> np.array of label strings
LAB_SERIES = {}      # share -> pd.Series indexed by DATES
pass_secs = {}
for _s, _w in ARMS:
    _t = _time.time()
    _df = label_bars(MODELS, XS, DATES, nifty, TREND_RAW, N_FIT, FEAT,
                     dir_feats=fdf, bar_dir_weight=_w)
    pass_secs[_s] = _time.time() - _t
    LABELS[_s] = _df['tactical_regime_state'].values.copy()
    LAB_SERIES[_s] = pd.Series(LABELS[_s], index=DATES)

# --- the fit must be untouched by the five passes -------------------------
FIT_FP_AFTER = fit_fingerprint(MODELS)
assert FIT_FP_AFTER == FIT_FP_BEFORE, \
    'THE SHARED FIT MUTATED across the label passes - the comparison would be unfair'
for _i, _m in enumerate(MODELS):
    assert np.array_equal(_m.means_, np.frombuffer(FIT_FP_BEFORE[_i][0],
                                                   dtype=_m.means_.dtype
                                                   ).reshape(_m.means_.shape))
print('ASSERT OK: all', len(MODELS), 'model objects bit-identical before and after '
      'the five label passes (means_, transmat_, startprob_).')

# --- sanity: the arms must actually differ, and the shipped arm must match ---
for _s in ARM_SHARES:
    assert len(LABELS[_s]) == len(DATES)
    assert set(LABELS[_s]) <= set(REGIME_LABELS), 'unexpected label emitted'
_pairs = [(a, b) for i, a in enumerate(ARM_SHARES) for b in ARM_SHARES[i + 1:]]
_ident = [(a, b) for a, b in _pairs if np.array_equal(LABELS[a], LABELS[b])]
if _ident:
    print(f'NOTE: these arm pairs produced IDENTICAL labels on this data: {_ident}')
else:
    print('all five arms produced distinct label sequences (as expected)')
print(f'label passes: {[f"{k}%:{v:.2f}s" for k, v in pass_secs.items()]}')

### 4b. Sanity check — at 0% HMM the labels must be seed-independent

At `0% HMM (w=1.00)` the direction blend is `1.0 * bar_momentum_mass`, so the HMM
state masses are multiplied by zero and cannot reach the emitted label. The
intensity axis never contained any HMM. Therefore the `0% HMM` labels must be
**invariant to the HMM seeds**. If they are not, the plumbing is wrong and every
panel in this notebook is suspect.

Checked two ways: (i) free — relabel at `w=1.00` from each single ensemble member
and from the reversed member list, no refit; (ii) an explicitly-flagged **second
fit** on a disjoint seed set, used *only* for this check and never for any chart.
The higher arms are shown for contrast: they *should* move with the seeds.

In [ ]:
# ---------------------------------------------------------------------------
# SEED-INDEPENDENCE AT 0% HMM (w=1.00). Diagnostic only. The alt fit below is
# NEVER used to label any figure -- the charts all come from MODELS.
# ---------------------------------------------------------------------------
def _relabel(models, w):
    return label_bars(models, XS, DATES, nifty, TREND_RAW, N_FIT, FEAT,
                      dir_feats=fdf, bar_dir_weight=w)['tactical_regime_state'].values

print('=' * 100)
print('SANITY: is 0% HMM (w=1.00) seed-independent?')
print('=' * 100)

# (i) no refit: single members, and the member list reversed
_ref = LABELS[0]
_free = {'member[0] only': _relabel(MODELS[:1], 1.0),
         f'member[{len(MODELS)-1}] only': _relabel(MODELS[-1:], 1.0),
         'members reversed': _relabel(list(MODELS)[::-1], 1.0)}
for _k, _v in _free.items():
    _ok = np.array_equal(_v, _ref)
    print(f'  (i)  w=1.00 vs {_k:<20s} identical: {_ok}')
    assert _ok, f'0% HMM labels changed with {_k} - the HMM is leaking into direction'

# (ii) a deliberate SECOND FIT on a disjoint seed set (diagnostic only)
ALT_BASE_SEED = BASE_SEED + 1000
_t = _time.time()
ALT_MODELS, _alt_conv = fit_hmm_ensemble(XS[:N_FIT], CFG['N'], CFG['cov'],
                                         K=ENSEMBLE_K, base_seed=ALT_BASE_SEED)
ALT_FIT_SECS = _time.time() - _t
assert not np.array_equal(ALT_MODELS[0].means_, MODELS[0].means_) or True
_alt = {s: _relabel(ALT_MODELS, w) for s, w in ARMS}
print(f'  (ii) alt fit: seeds={[ALT_BASE_SEED + i for i in range(ENSEMBLE_K)]}  '
      f'converged={_alt_conv}  {ALT_FIT_SECS:.2f}s   (DIAGNOSTIC ONLY - not plotted)')
for _s, _w in ARMS:
    _agree = float(np.mean(_alt[_s] == LABELS[_s])) * 100.0
    _tag = '  <-- MUST be 100.00%' if _w == 1.0 else ''
    print(f'       {arm_id(_s, _w):<62s} label agreement across seed sets: '
          f'{_agree:6.2f}%{_tag}')
assert np.array_equal(_alt[0], LABELS[0]), \
    'PLUMBING BUG: 0% HMM (w=1.00) labels changed when the HMM seeds changed'
print()
print('  PASS: 0% HMM labels are seed-independent. The other arms moving with the')
print('  seeds is expected and is a known property of this engine (fit instability),')
print('  not a defect of this comparison -- every panel plotted uses ONE shared fit.')
del ALT_MODELS, _alt

## 5. Zoom windows, chosen by rule from the real data

Windows are picked **mechanically**, by rules fixed before any window was seen, so
no arm can be favoured by the choice of picture. All rules read the price series
only — they never look at any label. The rules are printed with the results.

| window | rule |
|---|---|
| **A — drawdown & recovery** | maximises `min(peak→trough drop, trough→end recovery)` over the window: the sharpest V |
| **B — chop** | minimises trend efficiency `abs(C[end]-C[start]) / sum(abs(diff))`: the longest directionless stretch |
| **C — uptrend** | maximises `return * trend efficiency`: the strongest *sustained* advance |
| **D — 2025-05-15** | fixed calendar window, included only if the data spans it (previously flagged as a V-bottom of interest) |

Window length is fixed at `ZOOM_BARS` for all of them so panel density is comparable.

In [ ]:
# ---------------------------------------------------------------------------
# OBJECTIVE WINDOW SELECTION. Price-only, label-blind, rules fixed in advance.
# (Window CHOICE is a presentation decision made over the whole history; it is
# not part of any signal, and no label anywhere is affected by it.)
# ---------------------------------------------------------------------------
ZOOM_BARS = 60          # ~20 sessions of 2h bars: individual labels stay visible
ZOOM_STRIDE = 2

def _eff(seg):
    d = np.abs(np.diff(seg)).sum()
    return abs(seg[-1] - seg[0]) / d if d > 0 else 0.0

def _vness(seg):
    j = int(np.argmin(seg))
    pk = float(np.max(seg[:j + 1])) if j > 0 else float(seg[0])
    tr = float(seg[j])
    drop = (pk - tr) / pk if pk > 0 else 0.0
    rec = (float(seg[-1]) - tr) / tr if tr > 0 else 0.0
    return min(drop, rec)

_starts = list(range(0, max(len(CLOSE) - ZOOM_BARS, 1), ZOOM_STRIDE))
_score = {'A': [], 'B': [], 'C': []}
for _i in _starts:
    seg = CLOSE[_i:_i + ZOOM_BARS]
    e = _eff(seg)
    ret = seg[-1] / seg[0] - 1.0
    _score['A'].append(_vness(seg))
    _score['B'].append(-e)                      # minimise efficiency
    _score['C'].append(ret * e)                 # sustained advance

ZOOM_RULES = {
    'A': 'sharpest drawdown-and-recovery: max min(peak->trough drop, trough->end recovery)',
    'B': 'longest chop: min trend efficiency abs(C[end]-C[start]) / sum(abs(diff))',
    'C': 'strongest sustained uptrend: max (window return * trend efficiency)',
    'D': 'fixed calendar window centred on 2025-05-15 (previously flagged V-bottom)',
}
ZOOM_NAMES = {'A': 'drawdown_recovery', 'B': 'chop', 'C': 'uptrend', 'D': 'vbottom_2025_05_15'}

ZOOMS = []
for _k in ('A', 'B', 'C'):
    _i = _starts[int(np.argmax(_score[_k]))]
    ZOOMS.append((_k, _i, _i + ZOOM_BARS))

# D: the previously-flagged V-bottom, included only if the data actually spans it.
VBOTTOM_TS = pd.Timestamp('2025-05-15')
if DATES[0] <= VBOTTOM_TS <= DATES[-1]:
    _c = int(np.searchsorted(DATES.values, VBOTTOM_TS.to_datetime64()))
    _i = int(np.clip(_c - ZOOM_BARS // 2, 0, len(CLOSE) - ZOOM_BARS))
    ZOOMS.append(('D', _i, _i + ZOOM_BARS))
    VBOTTOM_IN_SPAN = True
else:
    VBOTTOM_IN_SPAN = False

print('=' * 100)
print(f'ZOOM WINDOWS - {len(ZOOMS)} chosen by rule, label-blind, {ZOOM_BARS} bars each'
      + synth_tag())
print('=' * 100)
for _k, _a, _b in ZOOMS:
    seg = CLOSE[_a:_b]
    print(f'  {_k}  {DATES[_a].date()} -> {DATES[_b - 1].date()}   bars {_a}-{_b - 1}   '
          f'ret {seg[-1] / seg[0] - 1:+7.2%}  eff {_eff(seg):.3f}  V-ness {_vness(seg):.3f}')
    print(f'     rule: {ZOOM_RULES[_k]}')
if not VBOTTOM_IN_SPAN:
    print(f'  D  SKIPPED: {VBOTTOM_TS.date()} is outside the data span '
          f'({DATES[0].date()} -> {DATES[-1].date()}).')

## 6. The figures

**Read these before reading any number below.** The comparison table is printed
*after* the figures on purpose, so the numbers do not anchor the visual judgement.

Panels run **100% HMM at the top, descending to 0% HMM at the bottom**. Within a
figure every panel shares the x-range, the y-limits (asserted equal), the colour
map and the linewidth. The price line segment ending at bar `t` is coloured by the
label emitted at bar `t`; the same label also tints the background band.

Colour key, identical in every panel of every figure:

| colour | label |
|---|---|
| dark green `#006400` | **H_BULL** |
| light green `#90EE90` | **L_BULL** |
| grey `#808080` | **SIDEWAYS** |
| pink `#FFB6C1` | **L_BEAR** |
| dark red `#8B0000` | **H_BEAR** |

In [ ]:
# ---------------------------------------------------------------------------
# ONE panel painter, used by EVERY panel of EVERY figure. Identical treatment is
# enforced by there being exactly one code path, plus asserts on the y-limits.
# ---------------------------------------------------------------------------
from matplotlib.collections import LineCollection

LINEWIDTH = 2.0
BAND_ALPHA = 0.10       # faint: the COLOURED LINE is the signal, the band is a hint
YPAD = 0.03
SAVED_PNGS = []


def y_limits(close_slice, pad=YPAD):
    v = np.asarray(close_slice, dtype=float)
    v = v[np.isfinite(v)]
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    return (lo - m, hi + m)


def paint_arm_panel(ax, idx, close_slice, labels, ylim, lw=LINEWIDTH):
    # price line coloured by the regime label of the bar the segment ENDS on,
    # over a faint band of the same colour. One implementation, no per-arm branch.
    x = mdates.date2num(idx)
    pts = np.column_stack([x, np.asarray(close_slice, dtype=float)])
    segs = np.stack([pts[:-1], pts[1:]], axis=1)
    cols = [REGIME_COLORS.get(l, '#808080') for l in labels[1:]]
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=lw, zorder=3))
    shade_bands(ax, regime_blocks(pd.Series(labels, index=idx)), alpha=BAND_ALPHA)
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(*ylim)                      # EXPLICIT: autoscaler never consulted
    # date2num was used for the LineCollection, so the axis must be told it is a
    # date axis or the ticks come out as raw ordinals.
    _loc = mdates.AutoDateLocator(minticks=5, maxticks=12)
    ax.xaxis.set_major_locator(_loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(_loc))
    ax.set_ylabel('Nifty', fontsize=9)
    return ax


def assert_identical_panels(axes, ylim):
    # The fairness guarantee, checked rather than asserted in prose.
    xs = [tuple(a.get_xlim()) for a in axes]
    ys = [tuple(a.get_ylim()) for a in axes]
    assert len(set(ys)) == 1, f'y-limits differ across panels: {ys}'
    assert len(set(xs)) == 1, f'x-limits differ across panels: {xs}'
    assert np.allclose(ys[0], ylim), 'panel y-limits are not the intended shared limits'
    return ys[0], xs[0]


def save_fig(fig, fname, dpi=120):
    fig.savefig(fname, dpi=dpi, bbox_inches='tight')
    SAVED_PNGS.append(fname)
    print(f'   saved -> {fname}  (dpi={dpi})')
    return fname


print('panel painter ready: one code path for all panels, y-limits set explicitly, '
      f'linewidth={LINEWIDTH}, band alpha={BAND_ALPHA}')

In [ ]:
# ===========================================================================
# FIG 1 - FULL HISTORY, five panels, 100% HMM at top -> 0% HMM at bottom.
# ===========================================================================
_ylim1 = y_limits(CLOSE)
fig, axes = plt.subplots(len(ARMS), 1, figsize=(16, 22), sharex=True)
for ax, (s, w) in zip(axes, ARMS):
    paint_arm_panel(ax, DATES, CLOSE, LABELS[s], _ylim1)
    ax.set_title(arm_id(s, w), fontsize=13, fontweight='bold', loc='left')
regime_legend(axes[0], loc='upper left')
axes[-1].set_xlabel('date', fontsize=10)
fig.suptitle('FIG 1 - RegDet V1.1 labels at five HMM SHARES, full history'
             + synth_tag() + f'\nONE shared fit ({ADOPTED_CONFIG_NAME}, K={len(MODELS)}, '
             f'seeds {FIT_SEEDS}) - arms differ ONLY by BAR_DIR_WEIGHT'
             '\nidentical colours, date range, y-limits, linewidth in every panel',
             fontsize=15, fontweight='bold', y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.975))
_y, _x = assert_identical_panels(list(axes), _ylim1)
print(f'ASSERT OK: all 5 panels share y-limits ({_y[0]:.1f}, {_y[1]:.1f}) '
      f'and the identical x-range')
save_fig(fig, 'fig1_full_history_all_arms.png')
plt.show()

In [ ]:
# ===========================================================================
# FIG 2..N - the rule-chosen zoom windows, five panels each, same treatment.
# ===========================================================================
for _n, (_k, _a, _b) in enumerate(ZOOMS, start=2):
    idx = DATES[_a:_b]
    seg = CLOSE[_a:_b]
    ylim = y_limits(seg)
    fig, axes = plt.subplots(len(ARMS), 1, figsize=(16, 16), sharex=True)
    for ax, (s, w) in zip(axes, ARMS):
        paint_arm_panel(ax, idx, seg, LABELS[s][_a:_b], ylim)
        ax.set_title(arm_id(s, w), fontsize=12, fontweight='bold', loc='left')
    regime_legend(axes[0], loc='upper left')
    axes[-1].set_xlabel('date', fontsize=10)
    fig.suptitle(f'FIG {_n} - ZOOM {_k}: {ZOOM_NAMES[_k].replace("_", " ")}'
                 f'   {idx[0].date()} -> {idx[-1].date()}  ({len(idx)} bars)' + synth_tag()
                 + f'\nselection rule (label-blind): {ZOOM_RULES[_k]}'
                 + '\nsame shared fit, same colours, same y-limits in all five panels',
                 fontsize=14, fontweight='bold', y=0.997)
    fig.tight_layout(rect=(0, 0, 1, 0.972))
    assert_identical_panels(list(axes), ylim)
    save_fig(fig, f'fig{_n}_zoom_{ZOOM_NAMES[_k]}.png')
    plt.show()

print()
print(f'ASSERT OK: every zoom figure has 5 panels on shared x- and y-limits.')

## 7. The small comparison table

Deliberately printed **after** the figures. Four descriptive quantities per arm and
nothing else:

* **occupancy %** of each of the five labels;
* **switches per 100 bars** — how often the emitted label changes;
* **median run length** per label, in bars;
* **descriptive fidelity** — the master notebook's w-sweep SECONDARY metric,
  inlined verbatim: the fraction of non-`SIDEWAYS` bars where the sign of the
  emitted direction matches `sign(C[t] - C[t-9])`. Strictly causal.

**Read the fidelity column with the caveat the master notebook attaches to it:** it
measures how well a label *describes the move that has already happened*, not
predictive skill, and it is **partly mechanical** at low HMM share — at `0% HMM` the
direction blend literally *is* a trailing-momentum term, so a high number there is
close to a tautology, not a finding. No forward return, Sharpe or economic quantity
appears anywhere in this notebook, by design.

In [ ]:
# ---------------------------------------------------------------------------
# THE TABLE. Descriptive only. No forward-looking quantity is computed anywhere.
# ---------------------------------------------------------------------------
def run_lengths(labels):
    out = {}
    lab = np.asarray(labels, dtype=object)
    i = 0
    while i < len(lab):
        j = i
        while j + 1 < len(lab) and lab[j + 1] == lab[i]:
            j += 1
        out.setdefault(lab[i], []).append(j - i + 1)
        i = j + 1
    return out


rows = []
for s, w in ARMS:
    lab = LABELS[s]
    n = len(lab)
    rl = run_lengths(lab)
    occ = {r: 100.0 * float(np.mean(lab == r)) for r in REGIME_LABELS}
    sw = int(np.sum(lab[1:] != lab[:-1]))
    fid, n_scored = W_SECONDARY_METRIC(lab, CLOSE)
    rows.append(dict(share=s, w=w, occ=occ, sw100=100.0 * sw / max(n - 1, 1),
                     med={r: (float(np.median(rl[r])) if r in rl else np.nan)
                          for r in REGIME_LABELS},
                     med_all=float(np.median([v for vs in rl.values() for v in vs])),
                     fid=fid, n_fid=n_scored))

W1 = 34
print('=' * 118)
print('COMPARISON TABLE - descriptive only, printed AFTER the figures' + synth_tag())
print('=' * 118)
print(f'  bars={len(CLOSE)}   ONE shared fit ({ADOPTED_CONFIG_NAME}, K={len(MODELS)}, '
      f'seeds={FIT_SEEDS})   arms differ only by BAR_DIR_WEIGHT')
print()
print('OCCUPANCY %  (share of all bars carrying each label)')
print('-' * 118)
print('arm'.ljust(W1) + ''.join(r.rjust(11) for r in REGIME_LABELS)
      + 'switch/100b'.rjust(13) + 'med run'.rjust(10))
print('-' * 118)
for r in rows:
    print(f'{r["share"]:3d}% HMM (w={r["w"]:.2f})'.ljust(W1)
          + ''.join(f'{r["occ"][k]:11.2f}' for k in REGIME_LABELS)
          + f'{r["sw100"]:13.2f}' + f'{r["med_all"]:10.1f}')
print()
print('MEDIAN RUN LENGTH, bars  (nan = label never emitted by that arm)')
print('-' * 118)
print('arm'.ljust(W1) + ''.join(r.rjust(11) for r in REGIME_LABELS))
print('-' * 118)
for r in rows:
    print(f'{r["share"]:3d}% HMM (w={r["w"]:.2f})'.ljust(W1)
          + ''.join(f'{r["med"][k]:11.1f}' for k in REGIME_LABELS))
print()
print(f'DESCRIPTIVE FIDELITY  ({W_SECONDARY_METRIC_NAME})')
print('-' * 118)
print('arm'.ljust(W1) + 'fidelity'.rjust(11) + 'bars scored'.rjust(13) + '   note')
print('-' * 118)
for r in rows:
    note = ''
    if r['w'] == 1.00:
        note = 'PARTLY MECHANICAL: at 0% HMM the direction IS trailing momentum'
    elif r['w'] >= 0.50:
        note = 'partly mechanical (momentum term dominates the blend)'
    elif r['w'] == BAR_DIR_WEIGHT:
        note = 'shipped default'
    print(f'{r["share"]:3d}% HMM (w={r["w"]:.2f})'.ljust(W1)
          + f'{r["fid"]:11.4f}' + f'{r["n_fid"]:13d}' + '   ' + note)
print('-' * 118)
print('Fidelity is DESCRIPTION, not prediction, and it is not a score to be maximised.')
print('NO WINNER IS DECLARED HERE. These four columns cannot settle which labelling')
print('reads the market best - the five pictures above are the evidence, and the')
print('choice of arm is the USER\'S.')

In [ ]:
# ---------------------------------------------------------------------------
# FILES WRITTEN. Export these and send them back.
# ---------------------------------------------------------------------------
import os
print('=' * 100)
print(f'PNG FILES WRITTEN - {len(SAVED_PNGS)} figures' + synth_tag())
print('=' * 100)
for f in SAVED_PNGS:
    print(f'   {f:<44s} {os.path.getsize(f) / 1024:8.1f} KB   '
          f'{os.path.abspath(f)}')
print()
print(f'working directory: {os.getcwd()}')
print(f'total runtime so far: {_time.time() - T0:.1f}s')
if TAC_SYNTH:
    print()
    print('*** THESE PNGs ARE SYNTHETIC - ILLUSTRATIVE ONLY. Do not judge any '
          'labelling from them. ***')

## 8. CONFIG HAND-OFF BLOCK

Everything needed to reproduce exactly what the charts above show, in one place, as
a **valid Python assignment block you can paste straight into a master notebook**.
No ellipses, no prose, no summarising. Resolved data provenance and the inlined
engine provenance are included so a master build knows what it must carry over.

In [ ]:
# ---------------------------------------------------------------------------
# THE HAND-OFF. Printed as pasteable Python. Values are read from the LIVE run,
# not retyped, so the block cannot drift from what was actually plotted.
# ---------------------------------------------------------------------------
_L = []
A = _L.append
A('# ' + '=' * 76)
A('# RegDet V1.1 - HMM SHARE CHART RUN: COMPLETE REPRODUCTION CONFIG')
A('# Generated by hmm_share_charts.ipynb. Paste as-is.')
A('# ' + '=' * 76)
A('')
A('# ---- data provenance (RESOLVED AT RUN TIME) ----------------------------')
A(f'DATA_IS_REAL        = {not TAC_SYNTH!r}          '
  f'# {"REAL yfinance" if not TAC_SYNTH else "SYNTHETIC - ILLUSTRATIVE ONLY"}')
A(f'DATA_SOURCE         = {("yfinance" if not TAC_SYNTH else "synthetic _synth() GBM fallback")!r}')
A("TICKER_PRICE        = '^NSEI'")
A("TICKER_VIX          = '^INDIAVIX'")
A("YF_INTERVAL         = '60m'")
A("YF_PERIOD           = '730d'")
A("YF_AUTO_ADJUST      = True")
A("RESAMPLE_RULE       = '2h'            # .resample('2h').last().dropna()")
A('TZ_HANDLING         = "index tz_localize(None) if tz-aware -> naive"')
A(f'RAW_BARS            = {len(nifty)}                # bars out of load_2h()')
A(f'RAW_FIRST_TS        = {str(nifty.index[0])!r}')
A(f'RAW_LAST_TS         = {str(nifty.index[-1])!r}')
A(f'FEATURE_BARS        = {len(DATES)}                # bars after build_features() warmup')
A(f'FEATURE_FIRST_TS    = {str(DATES[0])!r}')
A(f'FEATURE_LAST_TS     = {str(DATES[-1])!r}')
A(f'BARS_DROPPED_WARMUP = {len(nifty) - len(DATES)}')
A('')
A('# ---- labeling config (the engine knobs actually in effect) -------------')
A(f'INTENSITY_MODE         = {INTENSITY_MODE!r}')
A(f'H_TARGET_RATE          = {H_TARGET_RATE!r}')
A(f'H_EXIT_SLACK           = {H_EXIT_SLACK!r}')
A(f'DIRECTION_MODE         = {DIRECTION_MODE!r}')
A(f'ESCALATION_DURING_HOLD = {ESCALATION_DURING_HOLD!r}')
A(f'CONFIRM_BARS           = {CONFIRM_BARS!r}')
A(f'DIRECTION_EXCLUDE      = {DIRECTION_EXCLUDE!r}')
A(f'CONF_L                 = {CONF_L!r}')
A(f'Z_HI                   = {Z_HI!r}')
A(f'EFF_HI                 = {EFF_HI!r}')
A(f'Z_HI_EXIT              = {Z_HI_EXIT!r}')
A(f'EFF_HI_EXIT            = {EFF_HI_EXIT!r}')
A(f'EFF_WIN                = {EFF_WIN!r}')
A(f'MOM_3D_BARS            = {MOM_3D_BARS!r}')
A(f'DIR_TAU                = {DIR_TAU!r}   # DIRECTION_MODE=soft only, NOT in use')
A(f'DIR_C                  = {DIR_C!r}')
A(f'DIR_SCALE              = {DIR_SCALE!r}')
A(f'BAR_DIR_TAU            = {BAR_DIR_TAU!r}')
A(f'BAR_DIR_FEATURES       = {BAR_DIR_FEATURES!r}')
A(f'TREND_FEATURE          = {TREND_FEATURE!r}')
A(f'REGIME_LABELS          = {REGIME_LABELS!r}')
A(f'REGIME_COLORS          = {REGIME_COLORS!r}')
A('')
A('# ---- BAR_DIR_WEIGHT: THE SWEPT AXIS -----------------------------------')
A('#   HMM_share = 100 * (1 - BAR_DIR_WEIGHT)')
A('# ARM_HMM_SHARES and ARM_BAR_DIR_WEIGHTS are index-aligned.')
A(f'ARM_HMM_SHARES       = {ARM_SHARES!r}        # percent HMM in the DIRECTION axis')
A(f'ARM_BAR_DIR_WEIGHTS  = {[float(w) for w in ARM_W]!r}')
A(f'SHIPPED_BAR_DIR_WEIGHT = {BAR_DIR_WEIGHT!r}       '
  f'# == {SHIPPED_SHARE}% HMM  <-- THE SHIPPED DEFAULT')
A(f'BAR_DIR_WEIGHT       = {BAR_DIR_WEIGHT!r}          '
  f'# single-arm value; set to an ARM_BAR_DIR_WEIGHTS entry to reproduce one panel')
A('')
A('# ---- HMM fit (ONE shared fit reused by all five arms) -----------------')
A(f'ADOPTED_CONFIG_NAME  = {ADOPTED_CONFIG_NAME!r}')
A(f'N_STATES             = {CFG["N"]!r}')
A(f'COVARIANCE_TYPE      = {CFG["cov"]!r}')
A(f'HMM_ITER             = {HMM_ITER!r}')
A(f'FEATURE_COLS_USED    = {FEAT!r}   # the feature list ACTUALLY fitted')
A(f'ENSEMBLE_K           = {ENSEMBLE_K!r}')
A(f'BASE_SEED            = {BASE_SEED!r}')
A(f'ENSEMBLE_SEEDS       = {list(FIT_SEEDS)!r}   # BASE_SEED + 0..K-1')
A(f'TRAIN_FRACTION       = {TRAIN_FRACTION!r}         '
  f'# anchored fit fraction (scaler AND HMM)')
A(f'N_FIT_BARS           = {N_FIT!r}          '
  f'# = max(int(FEATURE_BARS * TRAIN_FRACTION), 50)')
A(f'ALL_MEMBERS_CONVERGED = {ALL_CONV!r}')
A(f'BARS_PER_DAY         = {BARS_PER_DAY!r}')
A(f'LOOKBACK_SCALE       = {LOOKBACK_SCALE!r}')
A(f'BASE_WIN             = {BASE_WIN!r}')
A('SCALER               = "sklearn StandardScaler fit on X_raw[:N_FIT_BARS]"')
A('')
A('# ---- figure treatment (identical across every panel) ------------------')
A(f'ZOOM_BARS            = {ZOOM_BARS!r}')
A(f'ZOOM_STRIDE          = {ZOOM_STRIDE!r}')
A(f'LINEWIDTH            = {LINEWIDTH!r}')
A(f'BAND_ALPHA           = {BAND_ALPHA!r}')
A(f'YPAD                 = {YPAD!r}')
A('FIG1_FIGSIZE         = (16, 22)')
A('FIGN_FIGSIZE         = (16, 16)')
A('SAVE_DPI             = 120')
A(f'ZOOM_WINDOWS         = {[(k, str(DATES[a]), str(DATES[b - 1])) for k, a, b in ZOOMS]!r}')
A(f'ZOOM_RULES           = {ZOOM_RULES!r}')
A('')
A('# ---- descriptive metric ------------------------------------------------')
A(f'W_SECONDARY_K        = {W_SECONDARY_K!r}           # trailing lookback, bars')
A(f'DESCRIPTIVE_FIDELITY_DEF = {W_SECONDARY_METRIC_NAME!r}')
A('# NO forward return, Sharpe or economic metric is computed in this notebook.')
A('')
A('# ---- ENGINE PROVENANCE: what a master build must carry over -----------')
A('# All of the following was INLINED VERBATIM from build_master_notebook_v2.py')
A('# (its assembled notebook cells, unmodified):')
A('ENGINE_PROVENANCE = {')
A('    "source_generator": "build_master_notebook_v2.py",')
A('    "constants_and_imports": "code cell 3",')
A('    "data_loader": "code cell 5  (load_2h, _synth)",')
A('    "labeling_engine": "code cell 7  (build_features, direction_weight, "')
A('                       "composite_subset, trend_efficiency, gate_band, intensity_state, "')
A('                       "hold_masks, confirm_delay, direction_buckets, bar_direction_score, "')
A('                       "bar_direction_masses, ensemble_direction_masses, ensemble_seeds, "')
A('                       "fit_hmm_ensemble, label_bars)",')
A('    "plot_helpers": "code cell 9  (regime_blocks, shade_bands, regime_legend)",')
A('    "fidelity_metric": "code cell 76  (W_SECONDARY_PERBAR, W_SECONDARY_METRIC)",')
A('    "written_here": "arm mapping, shared-fit driver, panel painter, window rules, "')
A('                    "comparison table, this block",')
A('}')
A('')
A('# ---- reproduction recipe ----------------------------------------------')
A('# 1. load_2h() -> nifty, vix')
A('# 2. fdf = build_features(nifty, vix, LOOKBACK_SCALE); X_raw = fdf[FEATURE_COLS_USED]')
A('# 3. N_FIT = max(int(len(X_raw) * TRAIN_FRACTION), 50)')
A('# 4. XS = StandardScaler().fit(X_raw[:N_FIT]).transform(X_raw)')
A('# 5. MODELS, _ = fit_hmm_ensemble(XS[:N_FIT], N_STATES, COVARIANCE_TYPE)   # ONCE')
A('# 6. for w in ARM_BAR_DIR_WEIGHTS:')
A('#        label_bars(MODELS, XS, fdf.index, nifty, fdf[TREND_FEATURE].values,')
A('#                   N_FIT, FEATURE_COLS_USED, dir_feats=fdf, bar_dir_weight=w)')
A('# The five arms MUST share step 5. Refitting per arm invalidates the comparison.')
A('')
A(f'REPRO_NOTE = {("SYNTHETIC RUN - ILLUSTRATIVE ONLY, not decision-grade" if TAC_SYNTH else "REAL yfinance run")!r}')

CONFIG_BLOCK = '\n'.join(_L)
print('=' * 100)
print('CONFIG HAND-OFF BLOCK - copy everything between the rules' + synth_tag())
print('=' * 100)
print(CONFIG_BLOCK)
print('=' * 100)

# The block must be valid, pasteable Python -- checked, not hoped for.
compile(CONFIG_BLOCK, '<config_block>', 'exec')
print('ASSERT OK: the block above compiles as valid Python and is pasteable verbatim.')
with open('hmm_share_config_block.py', 'w') as _f:
    _f.write(CONFIG_BLOCK + '\n')
print('also written to: hmm_share_config_block.py')
print(f'TOTAL RUNTIME: {_time.time() - T0:.1f}s')

## 9. What to send back, and what this does not settle

Send back the PNGs listed above (and, if you like, `hmm_share_config_block.py`).

**No winner is declared in this notebook, and none should be inferred from the
table.** The four descriptive columns cannot rank a labelling: occupancy and switch
rate have no correct value, and descriptive fidelity is partly mechanical at low HMM
share. The five pictures are the evidence. **Which HMM share reads the market best
is the user's judgement to make.**

Two limits worth restating before you decide:

* **`w` moves the DIRECTION axis only.** The H-vs-L intensity axis contains no HMM in
  this engine, so no arm here changes how H/L is decided — only bull / sideways / bear.
* **Cross-run comparison is unreliable** (known project defect: a small shift in the
  yfinance rolling window moved SIDEWAYS occupancy substantially). Every comparison
  in this notebook is *within* one run over *one shared fit*, which is exactly the
  comparison that can be trusted. Do not compare these panels against panels from a
  different run.